## LLM Priors Assessments

### Define functions to generate descriptions and priors for synthetic datasets

In [ ]:
import re
from pathlib import Path

import pandas as pd
import pyagrum as gum
from pydantic import BaseModel, Field
from typing import Annotated
from tqdm.asyncio import tqdm

from priors.llm import (
    GraphDescriptionBase,
    extract,
    parse_graph_description,
)
from priors.prompt import prepare_graph_description, prepare_priors

async def generate_graph_description(
    causal_graph: gum.BayesNet,
    model: str = "gemini-2.5-flash",
) -> GraphDescriptionBase:
    graph_prompt = prepare_graph_description(causal_graph)
    graph_description_raw = (
        (await extract(graph_prompt, None, model=model)).choices[0].message.content
    )
    assert graph_description_raw is not None, "Failed to obtain graph description"

    return await parse_graph_description(
        graph_description_raw, parse_method="llm", valid_vars=causal_graph.names()
    )


async def enrich_graph(
    causal_graph: gum.BayesNet,
    model: str = "gemini-2.5-flash",
    save_dir: str | Path | None = None,
):
    graph_description = await generate_graph_description(causal_graph, model=model)
    for name, description in graph_description.variable_descriptions.items():
        causal_graph.variableFromName(name).setDescription(description)
    causal_graph.setProperty("name", graph_description.title)  # pyagrum ignores the name property when loading from BIFXML
    if save_dir is not None:
        if isinstance(save_dir, str):
            save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / f"{graph_description.identifier}.bifxml"
        causal_graph.saveBIFXML(str(save_path))

    return graph_description

def extract_facts(string: str) -> list[dict]:
    pattern = re.compile(r"ext_((in)?dep)\((.+)\). I=([01].\d+), NA\n")
    matches = pattern.findall(string)
    facts = []
    for match in matches:
        cit_type, _, triple, score = match
        X, Y, S = triple.split(",")
        facts.append(
            {
                "cit_type": cit_type,
                "X": int(X),
                "Y": int(Y),
                "S": set() if S == "empty" else {int(var) for var in S[1:].split("y")},
                "score": float(score),
            }
        )
    return pd.DataFrame(facts).sort_values(
        by="score", ascending=False, ignore_index=True
    )

async def generate_priors(
    bn: gum.BayesNet,
    variable_descriptions: dict[str, str] | None = None,
    prior_model: str = "gemini-2.5-flash",
    parse_model: str = "gemini-2.5-flash-lite",
) -> dict:
    seed = 2025
    gum.initRandom(seed=seed)

    priors_prompt = prepare_priors(bn, descriptions=variable_descriptions)
    priors_raw = None
    while priors_raw is None:
        priors_raw = (
            (await extract(priors_prompt, None, model=prior_model)).choices[0].message.content
        )

    valid_var_pattern = (
        r"|".join(re.escape(var) for var in bn.names())
    )
    VarType = Annotated[str, Field(pattern=valid_var_pattern)]
    class VarConstraints(BaseModel):
        forbidden: set[tuple[VarType, VarType]] = set()
        required: set[tuple[VarType, VarType]] = set()

    priors = await extract(
        prompt=priors_raw, model=parse_model, pydantic_model=VarConstraints,
    )
    true_arrows = {
        (bn.variable(id1).name(), bn.variable(id2).name())
        for id1, id2 in bn.arcs()
    }

    return {
        "priors": priors.model_dump(mode="json"),
        "forbidden_Precision": len(priors.forbidden - true_arrows) / max(len(priors.forbidden), 1),
        "required_Precision": len(priors.required & true_arrows) / max(len(priors.required), 1),
    }


async def evaluate_priors(bif_paths: list[str | Path], prior_model: str, parse_model: str = None, exclude_descriptions: bool = False) -> pd.DataFrame:
    if parse_model is None:
        parse_model = prior_model
    prior_results = []
    prior_tasks = []
    for bif_path in bif_paths:
        if isinstance(bif_path, str):
            bif_path = Path(bif_path)
        bn = gum.loadBN(str(bif_path))
        variable_descriptions = {
            name: bn.variable(name).description() for name in bn.names()
        }
        if exclude_descriptions or not any(variable_descriptions.values()):
            variable_descriptions = None
        prior_results.append(
            {
                "bn": bn,
                "title": bn.propertyWithDefault("name", "no_name"),
                "filename": bif_path.stem,
                "num_nodes": bn.size(),
                "num_edges": len(bn.arcs()),
                "variable_descriptions": variable_descriptions,
            }
        )
        prior_tasks.append(
            generate_priors(
                bn=bn,
                variable_descriptions=variable_descriptions,
                prior_model=prior_model,
                parse_model=parse_model,
            )
        )
    prior_res = await tqdm.gather(*prior_tasks)
    for res_dict, prior_res in zip(prior_results, prior_res):
        res_dict.update(prior_res)
    df = pd.DataFrame(prior_results)

    return df

### Define experiment and helper function to evaluate the precision of LLM removed edges as causal priors

In [2]:
async def run_experiments(
    dataset_paths,
    n_runs=5,
    exclude_descriptions=True,
    prior_model="gemini-2.5-flash",
    parse_model=None,
):
    """
    Run multiple experiments and return combined results.

    Usage:
        results_no_desc = await run_experiments(bnlearn_small_datasets, n_runs=5, exclude_descriptions=True)
        results_with_desc = await run_experiments(bnlearn_small_datasets, n_runs=5, exclude_descriptions=False)
    """
    all_results = []

    for run_id in range(n_runs):
        print(f"Run {run_id + 1}/{n_runs}")
        df = await evaluate_priors(
            bif_paths=dataset_paths,
            prior_model=prior_model,
            parse_model=parse_model,
            exclude_descriptions=exclude_descriptions,
        )
        df["run_id"] = run_id
        all_results.append(df)

    return pd.concat(all_results, ignore_index=True)


def get_summary(df, metrics=["Precision", "Recall", "F-beta"]):
    """
    Get summary statistics for each dataset.

    Usage:
        summary = get_summary(results_no_desc)
    """
    summary_data = []

    for dataset in df["filename"].unique():
        dataset_df = df[df["filename"] == dataset]

        for metric in metrics:
            values = dataset_df[metric].dropna()

            if len(values) > 0:
                summary_data.append(
                    {
                        "Dataset": dataset,
                        "Metric": metric,
                        "Mean": values.mean(),
                        "Std": values.std(),
                        "Min": values.min(),
                        "Max": values.max(),
                        "Runs": len(values),
                    }
                )

    return pd.DataFrame(summary_data)


def show_report(df, metrics=["Precision", "Recall", "F-beta"]):
    """
    Get summary statistics for each dataset.

    Usage:
        summary = show_report(results_no_desc)
    """
    groups = df.groupby(["filename", "with_desc"])
    meta = groups.agg(
        {
            "num_nodes": "first",
            "num_edges": "first",
        }
    )
    meta["repeats"] = groups.size()
    meta.columns = pd.MultiIndex.from_product([["meta"], meta.columns])
    index = (
        groups["num_nodes"]
        .first()
        .reset_index()
        .sort_values(["num_nodes", "filename", "with_desc"])
        .set_index(["filename", "with_desc"])
        .index
    )
    summary = pd.concat([meta, groups[metrics].agg(["mean", "std"])], axis=1)
    return summary.loc[index]

### Experiment Configurations

In [ ]:
repeats = 5
prior_model = "gemini-2.5-flash"
parse_model = "gemini-2.5-flash-lite"


In [ ]:
synthetic_5_datasets = list(Path("causenet_synth_5_10_15").glob("*.bifxml"))
res_desc = await run_experiments(
    synthetic_5_datasets,
    n_runs=repeats,
    prior_model=prior_model,
    parse_model=parse_model,
    exclude_descriptions=False,
)
res_desc.drop(columns=["bn"]).to_json("results_synthetic_with_desc.json", orient="records", indent=4)
print("First half done")
res_desc

Run 1/5


100%|██████████| 54/54 [01:47<00:00,  2.00s/it]


Run 2/5


100%|██████████| 54/54 [01:28<00:00,  1.64s/it]


Run 3/5


...

100%|██████████| 54/54 [01:36<00:00,  1.78s/it]


Run 5/5


100%|██████████| 54/54 [01:47<00:00,  1.99s/it]

First half done


,bn,title,filename,num_nodes,num_edges,variable_descriptions,priors,forbidden_Precision,required_Precision,run_id
0,"BN{nodes: 5, arcs: 4, domainSize: 32, dim: 10,...",dag_5_nodes_5_edges_semantics_SF,dag_5_nodes_5_edges_semantics_SF,5,4,"{'irritability': 'irritability', 'tumors': 'tu...","{'forbidden': [['sleep_apnea', 'tumors'], ['dy...",1.000000,0.571429,0
1,"BN{nodes: 15, arcs: 14, domainSize: 32768, dim...",dag_15_nodes_15_edges_none_SF,dag_15_nodes_15_edges_none_SF,15,14,"{'pollution': 'pollution', 'cell_damage': 'cel...","{'forbidden': [['chest_pain', 'pollution'], ['...",1.000000,0.250000,0
2,"BN{nodes: 5, arcs: 7, domainSize: 32, dim: 17,...",dag_5_nodes_7_edges_none_random,dag_5_nodes_7_edges_none_random,5,7,{'ventricular_fibrillation': 'ventricular_fibr...,"{'forbidden': [['myocardial_infarction', 'coro...",1.000000,0.833333,0
3,"BN{nodes: 10, arcs: 9, domainSize: 1024, dim: ...",dag_10_nodes_10_edges_semantics_SF,dag_10_nodes_10_edges_semantics_SF,10,9,"{'blockages': 'blockages', 'premature_birth': ...","{'forbidden': [['premature_birth', 'alcoholism...",1.000000,0.500000,0
4,"BN{nodes: 15, arcs: 14, domainSize: 32768, dim...",dag_15_nodes_22_edges_semantics_SF,dag_15_nodes_22_edges_semantics_SF,15,14,"{'irritability': 'irritability', 'situations':...","{'forbidden': [['obsession', 'ear_infections']...",1.000000,0.235294,0
...,...,...,...,...,...,...,...,...,...,...
265,"BN{nodes: 5, arcs: 5, domainSize: 32, dim: 11,...",dag_5_nodes_5_edges_none_random,dag_5_nodes_5_edges_none_random,5,5,"{'discrimination': 'discrimination', 'substanc...","{'forbidden': [['misconduct', 'substance_abuse...",0.800000,0.375000,4
266,"BN{nodes: 5, arcs: 7, domainSize: 32, dim: 17,...",dag_5_nodes_7_edges_none_ER,dag_5_nodes_7_edges_none_ER,5,7,"{'angina': 'angina', 'high_cholesterol': 'high...","{'forbidden': [['high_blood_pressure', 'numbne...",0.909091,1.000000,4
267,"BN{nodes: 10, arcs: 9, domainSize: 1024, dim: ...",dag_10_nodes_10_edges_degrees_SF,dag_10_nodes_10_edges_degrees_SF,10,9,"{'vomiting': 'vomiting', 'migraines': 'migrain...","{'forbidden': [['lightheadedness', 'hypotensio...",1.000000,0.400000,4
268,"BN{nodes: 5, arcs: 4, domainSize: 32, dim: 10,...",dag_5_nodes_7_edges_degrees_SF,dag_5_nodes_7_edges_degrees_SF,5,4,"{'ulcers': 'ulcers', 'rosacea': 'rosacea', 'ey...","{'forbidden': [['eye_problems', 'high_blood_pr...",1.000000,0.666667,4


In [ ]:
# res2_desc = await run_experiments(
#     synthetic_5_datasets,
#     n_runs=repeats,
#     prior_model=prior_model,
#     parse_model=parse_model,
#     exclude_descriptions=True,
# )
# res2_desc.drop(columns=["bn"]).to_json("results_synthetic_5_without_desc.json", orient="records", indent=4)
# print("Second half done")
# res2_desc

### Run experiments on bnlearn datasets with/without variable descriptions

In [ ]:
bnlearn_small_datasets = list(Path("bnlearn/").glob("*.bifxml"))

In [ ]:
# bnlearn_results_no_desc = await run_experiments(
#     bnlearn_small_datasets,
#     n_runs=repeats,
#     exclude_descriptions=True,
#     prior_model=prior_model,
#     parse_model=parse_model,
# )
# # Save complete results for the record
# bnlearn_results_no_desc.drop(columns="bn").to_json("results/2025/llm/ew.json", orient="records", indent=4)
# # get_summary(bnlearn_results_no_desc)

Run 1/1


100%|██████████| 5/5 [00:50<00:00, 10.07s/it]


In [ ]:
bnlearn_results_with_desc = await run_experiments(
    bnlearn_small_datasets,
    n_runs=repeats,
    exclude_descriptions=False,
    prior_model=prior_model,
    parse_model=parse_model,
)
# Save complete results for the record
bnlearn_results_with_desc.drop(columns="bn").to_json("results_bnlearn_with_desc.json", orient="records", indent=4)

Run 1/5


  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:58<00:00, 11.68s/it]


Run 2/5


100%|██████████| 5/5 [00:52<00:00, 10.49s/it]


...

100%|██████████| 5/5 [00:44<00:00,  8.84s/it]


Run 4/5


100%|██████████| 5/5 [01:14<00:00, 14.90s/it]


Run 5/5


100%|██████████| 5/5 [00:53<00:00, 10.73s/it]


In [ ]:
from pathlib import Path

import pyagrum as gum
import pandas as pd

from priors.schema import Constraints


def aggregate_priors(priors_path: str, bifs_path: str):
    res = []
    df = pd.read_json(priors_path)
    for dataset in df["filename"].unique():
        bn = gum.loadBN(str(Path(bifs_path) / f"{dataset}.bifxml"))
        true_arrows = {
            (bn.variable(id1).name(), bn.variable(id2).name()) for id1, id2 in bn.arcs()
        }

        df_sub = df[df["filename"] == dataset]
        priors = [Constraints(**prior_dict) for prior_dict in df_sub["priors"]]
        majority_priors = Constraints(
            forbidden=set.intersection(*[priors.forbidden for priors in priors]),
            required=set.intersection(*[priors.required for priors in priors]),
        )

        forbidden_metrics = {
            "forbidden_length": len(majority_priors.forbidden),
            "forbidden_Precision": len(majority_priors.forbidden - true_arrows)
            / max(len(majority_priors.forbidden), 1),
            "forbidden_Recall": len(majority_priors.forbidden - true_arrows)
            / (bn.size() * (bn.size() - 1) - len(true_arrows)),
        }
        forbidden_metrics["forbidden_F1"] = (
            2
            * forbidden_metrics["forbidden_Precision"]
            * forbidden_metrics["forbidden_Recall"]
            / max(
                forbidden_metrics["forbidden_Precision"]
                + forbidden_metrics["forbidden_Recall"],
                1e-6,
            )
        )

        required_metics = {
            "required_length": len(majority_priors.required),
            "required_Precision": len(majority_priors.required & true_arrows)
            / max(len(majority_priors.required), 1),
            "required_Recall": len(majority_priors.required & true_arrows)
            / (len(true_arrows)),
        }
        required_metics["required_F1"] = (
            2
            * required_metics["required_Precision"]
            * required_metics["required_Recall"]
            / max(
                required_metics["required_Precision"]
                + required_metics["required_Recall"],
                1e-6,
            )
        )
        res.append(
            {
                **df_sub[
                    ["filename", "num_nodes", "num_edges", "variable_descriptions"]
                ]
                .iloc[0]
                .to_dict(),
                "priors": majority_priors.model_dump(mode="json"),
                **forbidden_metrics,
                **required_metics,
            }
        )

    return pd.DataFrame(res)


bnlearn_priors_aggregated = aggregate_priors(
    "results_bnlearn_with_desc.json", "bnlearn"
)
synthetic_priors_aggregated = aggregate_priors(
    "results_synthetic_with_desc.json", "causenet_synth_5_10_15"
)